# B4 — XLM-R full fine-tune (v2)
T4 GPU, ~15 min (v1 took 834 s). This is the efficiency comparison target for P1 — its trainable_params and wall_clock_sec ARE the claim-1 evidence.

In [ ]:
# cell 1 — environment (rerun if the session dies)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf repo
!git clone -b feature/dataset-v3 https://github.com/ruwini01/Sinhala_English_Code_Mixed_Sentiment_Analysis.git repo
%cd /content/repo/ml
!pip uninstall -y -q torchao

import os, shutil
SAVE = "/content/drive/MyDrive/Final_Reporing_Sentiment_Analysis/thesis_v2"
for sub in ("results", "checkpoints", "tokenized"):
    os.makedirs(f"{SAVE}/{sub}", exist_ok=True)

def bank(*paths, sub="results"):
    """Copy artifacts to Drive IMMEDIATELY — sessions die without warning."""
    for p in paths:
        if os.path.isdir(p):
            shutil.copytree(p, f"{SAVE}/{sub}/{os.path.basename(p)}", dirs_exist_ok=True)
        elif os.path.exists(p):
            shutil.copy2(p, f"{SAVE}/{sub}/")
        else:
            print("MISSING (not banked):", p)
    print("banked ->", f"{SAVE}/{sub}:", ", ".join(os.path.basename(p) for p in paths))

In [ ]:
# cell 2 — HF token from Colab Secrets (key icon in left sidebar, name: HF_TOKEN)
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# cell 3 — data: raw csv -> preprocess (splits are LOCKED in the repo)
import hashlib, os
RAW = "data/raw/singlish_mixed_sentiment_complete.csv"
os.makedirs("data/raw", exist_ok=True)

DRIVE_RAW = f"{SAVE}/singlish_mixed_sentiment_complete.csv"
if os.path.exists(DRIVE_RAW):
    shutil.copy2(DRIVE_RAW, RAW)
else:
    from google.colab import files
    files.upload()                      # pick the raw CSV from your PC
    os.replace("singlish_mixed_sentiment_complete.csv", RAW)
    shutil.copy2(RAW, DRIVE_RAW)        # bank the raw file itself

sha = hashlib.sha256(open(RAW, "rb").read()).hexdigest()
assert sha == "5ca2952ebf1087dbc0704beb4c53306bf3e30ab201ff72a60171565a932ba221", f"WRONG RAW FILE — sha256 {sha[:16]}... != v2.1 (see ml/DATA.md)"
print("raw sha256 verified: v2.1")

!python -m src.preprocess.quarantine
!python -m src.preprocess.clean_text
!python -m src.preprocess.language_id

In [ ]:
# cell 4 — B4 run (T4, ~15 min)
!python -m src.train.run_xlmr_full
bank("results/xlmr_full.json")

In [ ]:
# cell last — download to PC (put into ml/results/ locally, then commit)
from google.colab import files
files.download("results/xlmr_full.json")